In [1]:
import os
from pathlib import Path
from copy import deepcopy

from pptx import Presentation

In [ ]:
# functions 
def copy_slide(source_prs, slide, dest_prs):
    """
    Copy a slide from one presentation to another.
    """
    # Create a blank slide using the destination presentation's
    # first layout (layout choice doesn't matter because we'll
    # copy all shapes over).
    blank_layout = dest_prs.slide_layouts[6]
    new_slide = dest_prs.slides.add_slide(blank_layout)

    # Remove any placeholder shapes
    for shape in list(new_slide.shapes):
        sp = shape.element
        sp.getparent().remove(sp)

    # Copy all shapes
    for shape in slide.shapes:
        new_slide.shapes._spTree.insert_element_before(
            deepcopy(shape.element),
            "p:extLst"
        )

    return new_slide


def merge_presentations(input_folder, output_file):
    input_folder = Path(input_folder)

    ppt_files = sorted(
        [f for f in input_folder.glob("*.pptx")
         if f.name != Path(output_file).name],
        key=lambda p: p.name.lower()
    )

    if not ppt_files:
        raise FileNotFoundError("No PowerPoint files found.")

    merged_prs = Presentation()

    # Remove the default blank slide
    if len(merged_prs.slides) > 0:
        slide_id = merged_prs.slides._sldIdLst[0]
        merged_prs.slides._sldIdLst.remove(slide_id)

    for ppt_file in ppt_files:
        print(f"Adding: {ppt_file.name}")

        prs = Presentation(ppt_file)

        for slide in prs.slides:
            copy_slide(prs, slide, merged_prs)

    merged_prs.save(output_file)
    print(f"Merged presentation saved to: {output_file}")


In [ ]:
INPUT_FOLDER = r"C:\Presentations"
OUTPUT_FILE = r"C:\Presentations\merged.pptx"

merge_presentations(INPUT_FOLDER, OUTPUT_FILE)